<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/fabiobento/dnn-course-2026-1/blob/main/C1_M4_Lab_1_cnn_nature_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
  </td>
</table>

# Construindo uma CNN para Classificação da Natureza

Suponha que o pessoal da área de biologia do campus nos procurou com uma ideia: um aplicativo que possa classificar não apenas flores, mas também insetos e pequenos animais. Sua tarefa é projetar e construir o modelo que tornará isso possível.

Este problema é mais complexo do que os que você enfrentou anteriormente. As camadas lineares que você utilizou antes não serão suficientes para capturar os ricos padrões visuais nestas diversas imagens. Para atender a este novo desafio, você construirá uma **Rede Neural Convolucional (CNN)**, um modelo projetado para reconhecer formas, texturas e características em dados visuais.

Neste laboratório, você passará pelo processo de ponta a ponta (*end-to-end*) de construção de uma CNN para esta tarefa de classificação. Você não apenas implementará a arquitetura, mas também seguirá um fluxo de trabalho iterativo — começando com um protótipo menor antes de aumentar a escala (*scaling up*) — e aprenderá a diagnosticar problemas comuns de treinamento.

Você irá:

* **Preparar um Conjunto de Dados Diversificado**: Carregar e transformar um subconjunto especializado de imagens para o seu classificador de natureza multiclasse.
* **Construir uma Arquitetura CNN**: Definir uma CNN completa do zero, combinando camadas convolucionais, de agrupamento (*pooling*) e totalmente conectadas (*fully connected*) para criar um poderoso extrator de características (*feature extractor*).
* **Treinar um Modelo Protótipo**: Seguir um fluxo de trabalho realista treinando primeiro o seu modelo em um subconjunto menor, de 9 classes, para construir um protótipo funcional e estabelecer uma linha de base (*baseline*) de desempenho.
* **Aumentar a Escala e Diagnosticar Desafios**: Treinar o modelo completo em todas as 15 classes e analisar os resultados para identificar desafios comuns de aprendizado de máquina, como o *overfitting* (sobreajuste).

## Importação de bibliotecas

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

import helper_utils

In [ ]:
# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Preparando o Conjunto de Dados da Natureza

Para este laboratório, você trabalhará com uma coleção de imagens retiradas do conhecido [conjunto de dados CIFAR-100](https://docs.pytorch.org/vision/main/generated/torchvision.datasets.CIFAR100.html). Este conjunto de dados é um recurso fantástico para tarefas de visão computacional, contendo milhares de pequenas **imagens coloridas de 32x32**, que são perfeitas para treinar uma CNN. É uma coleção diversificada, o que é exatamente o que você precisa para o aplicativo expandido de classificação da natureza.

Embora o CIFAR-100 tenha 100 classes diferentes, você não precisará de todas elas. Para atender aos novos requisitos do seu aplicativo, você se concentrará em uma seleção curada de **15 classes** que se encaixam no tema de um classificador de natureza. Esta seleção incluirá flores, insetos e mamíferos. Especificamente, você trabalhará com:

* **Flores** (*Flowers*): 'orchid', 'poppy', 'rose', 'sunflower', 'tulip'
* **Mamíferos** (*Mammals*): 'fox', 'porcupine', 'possum', 'raccoon', 'skunk'
* **Insetos** (*Insects*): 'bee', 'beetle', 'butterfly', 'caterpillar', 'cockroach'

### Transformações de Imagem

Antes de carregar o conjunto de dados, primeiro você precisa definir os pipelines de transformação para ele. Como todas as imagens no conjunto de dados já possuem um tamanho padrão de **32x32**, você não precisa adicionar uma etapa de redimensionamento. Seu pipeline de treinamento incluirá aumento de dados (*data augmentation*), enquanto ambos os pipelines converterão as imagens em **tensores** e as **normalizarão** usando os valores padrão de média e desvio padrão para o conjunto de dados CIFAR-100.

* Defina a média e o desvio padrão específicos para o conjunto de dados CIFAR-100.

In [ ]:
cifar100_mean = (0.5071, 0.4867, 0.4408)
cifar100_std = (0.2675, 0.2565, 0.2761)

* Defina dois pipelines separados utilizando `transforms.Compose`.
    * Um para o conjunto de treinamento, que inclui aumento de dados (*data augmentation*), e outro para o conjunto de validação.

In [ ]:
# Training set transformation pipeline
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(cifar100_mean, cifar100_std)
])

# Validation set transformation pipeline
val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(cifar100_mean, cifar100_std)
])

### Preparando o Pipeline de Dados

Com suas transformações prontas, é hora de carregar os dados. Para mostrar rapidamente um protótipo funcional ao borboletário vizinho, uma estratégia inteligente é começar com um conjunto de dados menor e mais gerenciável. Isso permite que você teste todo o seu pipeline e construa um modelo de linha de base (*baseline*) sem os longos tempos de espera exigidos para o conjunto de dados completo.

Portanto, em vez de usar todas as 15 classes de uma vez, você começará com um subconjunto balanceado de **9 classes** (**3 de cada categoria**). Essa abordagem iterativa é uma prática comum e eficiente no aprendizado de máquina do mundo real.

Para este protótipo inicial, você usará as seguintes classes:

* **Flores** (*Flowers*): 'orchid', 'poppy', 'sunflower'
* **Mamíferos** (*Mammals*): 'fox', 'raccoon', 'skunk'
* **Insetos** (*Insects*): 'butterfly', 'caterpillar', 'cockroach'

* Crie uma lista em Python contendo os nomes das 9 classes que você usará para o protótipo inicial.

In [ ]:
subset_target_classes = [
    # Flowers
    'orchid', 'poppy', 'sunflower',
    # Mammals
    'fox', 'raccoon', 'skunk',
    # Insects
    'butterfly', 'caterpillar', 'cockroach'
]

* Use a função auxiliar `load_cifar100_subset`, passando a sua lista `subset_target_classes` e ambos os pipelines de transformação.
* Esta função lida com todo o processo de carregamento: ela faz o download do conjunto de dados CIFAR-100 completo, aplica as transformações especificadas e, em seguida, filtra o resultado para incluir apenas as **9 classes** que você selecionou.
* Ela retorna os objetos finais dos conjuntos de dados de treinamento e validação, prontos para a próxima etapa.

In [ ]:
# Call the helper function to prepare the datasets
train_dataset_proto, val_dataset_proto = helper_utils.load_cifar100_subset(subset_target_classes, train_transform, val_transform)

* Com seus objetos `Dataset` prontos, a etapa final no pipeline de dados é criar os `DataLoaders`.

In [ ]:
# Set the number of samples to be processed in each batch
batch_size = 64

# Create a data loader for the training set, with shuffling enabled
train_loader_proto = DataLoader(train_dataset_proto, batch_size=batch_size, shuffle=True)

# Create a data loader for the validation set, without shuffling
val_loader_proto = DataLoader(val_dataset_proto, batch_size=batch_size, shuffle=False)

### Visualizando as Imagens de Treinamento

Com o seu pipeline de dados concluído, é sempre uma boa ideia analisar alguns exemplos do seu conjunto de treinamento. Isso ajuda a confirmar se os seus dados foram carregados e processados corretamente. A função auxiliar a seguir exibirá uma amostra aleatória das suas imagens de treinamento.

In [ ]:
# Visualize a 3x3 grid of random training images
helper_utils.visualise_images(train_dataset_proto, grid=(3, 3))

## Construindo a Arquitetura da CNN

Com seus dados prontos, é hora de construir o núcleo do seu classificador de natureza. Para uma tarefa complexa como a identificação de diferentes espécies em imagens, as camadas lineares que você usou antes não são suficientes, pois elas analisam os pixels individualmente sem entender suas relações espaciais.

Agora você construirá uma **Rede Neural Convolucional (CNN)**, uma arquitetura projetada especificamente para "ver" e reconhecer padrões, bordas e texturas em imagens através de uma série de filtros aprendíveis (*learnable filters*). Você definirá a estrutura do seu modelo usando o `nn.Module` do PyTorch, combinando vários tipos de camadas para criar um poderoso classificador de imagens.

Aqui está um detalhamento das principais camadas que você usará:

**Camada Convolucional ([nn.Conv2d](https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv2d.html))**

> Este é o bloco de construção central de uma CNN, que usa filtros aprendíveis para varrer a imagem em busca de características visuais. A saída é um conjunto de "mapas de características" (*feature maps*) que destacam onde esses padrões aparecem na imagem.
> * `in_channels`: O número de canais da camada anterior; para a primeira camada, este valor é 3, correspondente aos canais de cores RGB.
> * `out_channels`: O número de filtros que a camada irá aprender, determinando o número de mapas de características de saída.
> * `kernel_size`: As dimensões do filtro, como uma grade 3x3 que examina um pixel e seus vizinhos imediatos.
> * `padding`: Adiciona uma borda ao redor da imagem, permitindo que o *kernel* processe os pixels da extremidade enquanto preserva as dimensões originais da imagem.
> 
> 

**Função de Ativação ReLU ([nn.ReLU](https://docs.pytorch.org/docs/stable/generated/torch.nn.ReLU.html))**

> Uma função de ativação que introduz não-linearidade mudando todos os valores negativos nos mapas de características para zero. Isso ajuda o modelo a aprender padrões mais complexos.

**Camada de Agrupamento Máximo (*Max Pooling*) ([nn.MaxPool2d](https://docs.pytorch.org/docs/stable/generated/torch.nn.MaxPool2d.html))**

> Esta camada reduz a resolução (*downsamples*) dos mapas de características diminuindo sua altura e largura, o que torna a rede mais eficiente. Ela desliza uma janela sobre o mapa de características e mantém apenas o maior valor único daquela janela, descartando o resto.
> * `kernel_size`: O tamanho da janela na qual o agrupamento (*pooling*) será realizado, como uma área 2x2.
> * `stride`: O tamanho do passo que a janela se move através da imagem. Um *stride* de 2 com um *kernel* 2x2 reduzirá pela metade as dimensões do mapa de características.
> 
> 

**Camada de Achatamento (*Flatten*) ([nn.Flatten](https://docs.pytorch.org/docs/stable/generated/torch.nn.Flatten.html))**

> Uma camada utilitária que desenrola (*unrolls*) os mapas de características 2D em um único vetor 1D. Esta é uma etapa necessária para preparar os dados para as camadas lineares totalmente conectadas.

**Camada Linear ([nn.Linear](https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html))**

> Também conhecida como camada totalmente conectada (*fully connected layer*), ela realiza a classificação final. Ela combina as características aprendidas pelas camadas convolucionais em uma previsão final.

**Camada de Abandono (*Dropout*) ([nn.Dropout](https://docs.pytorch.org/docs/stable/generated/torch.nn.Dropout.html))**

> Uma técnica de regularização que ajuda a evitar o *overfitting* (sobreajuste), definindo aleatoriamente uma fração das ativações dos neurônios como zero durante o treinamento. Isso força a rede a aprender características mais robustas em vez de depender muito de qualquer padrão único.

In [ ]:
class SimpleCNN(nn.Module):
    """
    A simple Convolutional Neural Network model.

    The architecture consists of three convolutional blocks followed by two
    fully connected layers for classification.
    """
    def __init__(self, num_classes):
        """
        Initializes the layers of the neural network.

        Args:
            num_classes: The number of output classes for the final layer.
        """
        # Call the constructor of the parent class (nn.Module)
        super(SimpleCNN, self).__init__()
        
        # Define the first convolutional block
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Define the second convolutional block
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Define the third convolutional block
        self.conv3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
        self.relu3 = nn.ReLU()
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Define the layer to flatten the feature maps
        self.flatten = nn.Flatten()

        # Define the fully connected (dense) layers
        # Input image is 32x32, after 3 pooling layers: 4x4
        self.fc1 = nn.Linear(128 * 4 * 4, 512)
        self.relu4 = nn.ReLU()
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(512, num_classes)


    def forward(self, x):
        """
        Defines the forward pass of the model.

        Args:
            x: The input tensor of shape (batch_size, channels, height, width).

        Returns:
            The output tensor containing the logits for each class.
        """
        # Pass input through the first convolutional block
        x = self.conv1(x)
        x = self.relu1(x)
        x = self.pool1(x)

        # Pass feature maps through the second convolutional block
        x = self.conv2(x)
        x = self.relu2(x)
        x = self.pool2(x)
        
        # Pass feature maps through the third convolutional block
        x = self.conv3(x)
        x = self.relu3(x)
        x = self.pool3(x)

        # Flatten the output for the fully connected layers
        x = self.flatten(x)

        # Pass the flattened features through the fully connected layers
        x = self.fc1(x)
        x = self.relu4(x)
        x = self.dropout(x)
        x = self.fc2(x)

        # Return the final output logits
        return x

Com a arquitetura `SimpleCNN` definida, o próximo passo é criar uma instância do modelo para o seu protótipo.

* Primeiro, determine dinamicamente o número de classes de saída verificando o comprimento da lista de classes no seu `train_dataset_proto`.
* Crie uma instância da sua `SimpleCNN`, passando `num_classes` para o seu construtor.

In [ ]:
# Get the number of classes
num_classes = len(train_dataset_proto.classes)

# Instantiate the model
prototype_model = SimpleCNN(num_classes)

Antes de começar o treinamento, é muito útil visualizar como o formato dos seus dados muda à medida que eles passam pela CNN. Isso confirmará que a sua arquitetura está configurada corretamente e mostrará como as dimensões espaciais encolhem enquanto o número de canais aumenta a cada bloco convolucional.

* Defina a função auxiliar `print_data_flow`.
* Esta função passará uma imagem colorida de amostra de 32x32 pelo seu modelo, camada por camada, imprimindo o formato (*shape*) do tensor em cada etapa principal para rastrear a sua jornada, desde a entrada até a previsão final.

In [ ]:
def print_data_flow(model):
    """
    Prints the shape of a tensor as it flows through each layer of the model.

    Args:
        model: An instance of the PyTorch model to inspect.
    """
    # Create a sample input tensor (batch_size, channels, height, width)
    x = torch.randn(1, 3, 32, 32)

    # Track the tensor shape at each stage
    print(f"Input shape: \t\t{x.shape}")

    # First conv block
    x = model.conv1(x)
    print(f"After conv1: \t\t{x.shape}")
    x = model.relu1(x)
    x = model.pool1(x)
    print(f"After pool1: \t\t{x.shape}")

    # Second conv block
    x = model.conv2(x)
    print(f"After conv2: \t\t{x.shape}")
    x = model.relu2(x)
    x = model.pool2(x)
    print(f"After pool2: \t\t{x.shape}")

    # Third conv block
    x = model.conv3(x)
    print(f"After conv3: \t\t{x.shape}")
    x = model.relu3(x)
    x = model.pool3(x)
    print(f"After pool3: \t\t{x.shape}")

    # Flatten using the model's flatten layer
    x = model.flatten(x)
    print(f"After flatten: \t\t{x.shape}")

    # Fully connected layers
    x = model.fc1(x)
    print(f"After fc1: \t\t{x.shape}")
    x = model.relu4(x)
    x = model.dropout(x)
    x = model.fc2(x)
    print(f"Output shape (fc2): \t{x.shape}")

Você agora pode imprimir um resumo do seu modelo e rastrear o fluxo de dados para vê-lo em ação.

* Chame a sua função auxiliar para imprimir o formato (*shape*) do tensor em cada etapa.
* O tensor começa como uma imagem `(1, 3, 32, 32)`. À medida que passa pelos blocos `conv` e `pool`, o número de **canais aumenta** enquanto o **tamanho espacial é reduzido pela metade** a cada etapa.
* O mapa de características final `(1, 128, 4, 4)` é **achatado** (*flattened*) em um vetor 1D para ser processado pelas camadas lineares. O **formato de saída** final do modelo **é** `(1, 9)`, fornecendo uma pontuação para cada uma das 9 classes.

In [ ]:
# Print the model's architecture
print(prototype_model)

# Call the helper function to visualize the data flow
print("\n--- Tracing Data Flow ---")
print_data_flow(prototype_model)

## Treinando o Modelo

Com o seu modelo definido e o pipeline de dados preparado, você está pronto para configurar o processo de treinamento. Isso envolve inicializar uma função de perda para medir o erro do seu modelo e um otimizador para atualizar os seus pesos com base nesse erro.

### Inicializar a Função de Perda e o Otimizador

Antes de iniciar o loop de treinamento, você definirá dois componentes principais:

* Você usará [nn.CrossEntropyLoss](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html). Esta é a função de perda padrão para tarefas de classificação multiclasse, pois é projetada para medir o erro quando um modelo precisa escolher uma classe entre várias possibilidades.
* Você usará o otimizador [Adam](https://docs.pytorch.org/docs/stable/generated/torch.optim.Adam.html). Este é um algoritmo popular e eficiente que atualiza os pesos do modelo para minimizar a perda.

In [ ]:
# Loss function
loss_function = nn.CrossEntropyLoss()

# Optimizer for the prototype model
optimizer_prototype = optim.Adam(prototype_model.parameters(), lr=0.001)

### O Loop de Treinamento

* Em seguida, você definirá a função `training_loop`. Esta função encapsula todo o processo de treinamento e validação do seu modelo ao longo de múltiplas épocas.

In [ ]:
def training_loop(model, train_loader, val_loader, loss_function, optimizer, num_epochs, device):
    """
    Trains and validates a PyTorch neural network model.

    Args:
        model: The neural network model to be trained.
        train_loader: DataLoader for the training dataset.
        val_loader: DataLoader for the validation dataset.
        loss_function: The loss function to use for training.
        optimizer: The optimization algorithm.
        num_epochs: The total number of epochs to train for.
        device: The device (e.g., 'cpu' or 'cuda') to run the training on.

    Returns:
        A tuple containing:
        - The trained model.
        - A list of metrics [train_losses, val_losses, val_accuracies].
    """
    # Move the model to the specified device (CPU or GPU)
    model.to(device)
    
    # Initialize lists to store training and validation metrics
    train_losses = []
    val_losses = []
    val_accuracies = []
    
    # Print a message indicating the start of the training process
    print("--- Training Started ---")
    
    # Loop over the specified number of epochs
    for epoch in range(num_epochs):
        # Set the model to training mode
        model.train()
        # Initialize running loss for the current epoch
        running_loss = 0.0
        # Iterate over batches of data in the training loader
        for images, labels in train_loader:
            # Move images and labels to the specified device
            images, labels = images.to(device), labels.to(device)
            
            # Clear the gradients of all optimized variables
            optimizer.zero_grad()
            # Perform a forward pass to get model outputs
            outputs = model(images)
            # Calculate the loss
            loss = loss_function(outputs, labels)
            # Perform a backward pass to compute gradients
            loss.backward()
            # Update the model parameters
            optimizer.step()
            
            # Accumulate the training loss for the batch
            running_loss += loss.item() * images.size(0)
            
        # Calculate the average training loss for the epoch
        epoch_loss = running_loss / len(train_loader.dataset)
        # Append the epoch loss to the list of training losses
        train_losses.append(epoch_loss)
        
        # Set the model to evaluation mode
        model.eval()
        # Initialize running validation loss and correct predictions count
        running_val_loss = 0.0
        correct = 0
        total = 0
        # Disable gradient calculations for validation
        with torch.no_grad():
            # Iterate over batches of data in the validation loader
            for images, labels in val_loader:
                # Move images and labels to the specified device
                images, labels = images.to(device), labels.to(device)
                
                # Perform a forward pass to get model outputs
                outputs = model(images)
                
                # Calculate the validation loss for the batch
                val_loss = loss_function(outputs, labels)
                # Accumulate the validation loss
                running_val_loss += val_loss.item() * images.size(0)
                
                # Get the predicted class labels
                _, predicted = torch.max(outputs, 1)
                # Update the total number of samples
                total += labels.size(0)
                # Update the number of correct predictions
                correct += (predicted == labels).sum().item()
                
        # Calculate the average validation loss for the epoch
        epoch_val_loss = running_val_loss / len(val_loader.dataset)
        # Append the epoch validation loss to the list
        val_losses.append(epoch_val_loss)
        
        # Calculate the validation accuracy for the epoch
        epoch_accuracy = 100.0 * correct / total
        # Append the epoch accuracy to the list
        val_accuracies.append(epoch_accuracy)
        
        # Print the metrics for the current epoch
        print(f"Epoch [{epoch+1}/{num_epochs}], Train Loss: {epoch_loss:.4f}, Val Loss: {epoch_val_loss:.4f}, Val Accuracy: {epoch_accuracy:.2f}%")
        
    # Print a message indicating the end of the training process
    print("--- Finished Training ---")
    
    # Consolidate all metrics into a single list
    metrics = [train_losses, val_losses, val_accuracies]
    
    # Return the trained model and the collected metrics
    return model, metrics

Com todos os componentes no lugar, você está pronto para iniciar o treinamento.

* Execute a função `training_loop` com o seu modelo protótipo (para 9 classes), seus respectivos *data loaders* (carregadores de dados), a função de perda e o otimizador.
* Você treinará por `15 epochs` (épocas), e a função retornará o modelo treinado junto com as métricas de desempenho coletadas.
* Após a conclusão do treinamento, você usará a função auxiliar `plot_training_metrics` para visualizar a perda (*loss*) de treinamento e de validação, juntamente com a acurácia de validação.

In [ ]:
# Start the training process by calling the training loop function
trained_proto_model, training_metrics_proto = training_loop(
    model=prototype_model, 
    train_loader=train_loader_proto, 
    val_loader=val_loader_proto, 
    loss_function=loss_function, 
    optimizer=optimizer_prototype, 
    num_epochs=15, 
    device=device
)

# Visualize the training metrics (loss and accuracy)
print("\n--- Training Plots ---\n")
helper_utils.plot_training_metrics(training_metrics_proto)

Excelente trabalho! O modelo protótipo está treinado e os resultados parecem muito promissores. Alcançar uma acurácia de validação superior a **75%** no subconjunto de 9 classes é um ótimo resultado e confirma que a sua arquitetura CNN é muito adequada para esta tarefa.

Este protótipo bem-sucedido lhe dá o sinal verde para avançar para a próxima fase: treinar um modelo completo em todas as 15 classes para o borboletário. Mas, antes de fazer isso, é útil realizar uma última verificação qualitativa para ver como o seu modelo "pensa".

### Visualizando as Previsões

Embora os gráficos mostrem o desempenho geral do seu modelo, observar previsões individuais proporciona uma noção mais intuitiva dos seus pontos fortes e fracos. Agora você pode usar uma função auxiliar para ver o seu modelo em ação, visualizando suas previsões em imagens aleatórias do conjunto de validação. Isso mostrará exemplos concretos de onde ele acerta e onde pode estar cometendo erros.

In [ ]:
# Visualize model predictions on a sample of validation images
helper_utils.visualise_predictions(
    model=trained_proto_model, 
    data_loader=val_loader_proto, 
    device=device, 
    grid=(3, 3)
)

## Aumentando a Escala: Treinando o Modelo Completo

O protótipo foi um sucesso! Agora é hora de treinar o modelo final para o aplicativo do borboletário. Você repetirá os mesmos passos de antes, mas desta vez usando o conjunto de dados completo e mais desafiador de **15 classes**.

* Primeiro, crie uma nova lista contendo todas as 15 classes alvo.
* Use a função auxiliar `load_cifar100_subset` novamente para criar os novos conjuntos de dados de treinamento e validação com base nesta lista completa.

In [ ]:
# Define the full class list.
all_target_classes = [
    # Flowers
    'orchid', 'poppy', 'rose', 'sunflower', 'tulip',
    # Mammals
    'fox', 'porcupine', 'possum', 'raccoon', 'skunk',
    # Insects
    'bee', 'beetle', 'butterfly', 'caterpillar', 'cockroach'
]

# Load the full datasets.
train_dataset, val_dataset = helper_utils.load_cifar100_subset(all_target_classes, train_transform, val_transform)

* Encapsule seus novos conjuntos de dados de 15 classes em instâncias de `DataLoader`, utilizando o mesmo `batch_size=64`.

In [ ]:
# Create a data loader for the training set, with shuffling enabled
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# Create a data loader for the validation set, without shuffling
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

* Exiba uma amostra de imagens do seu novo conjunto de treinamento de 15 classes para confirmar que ele foi carregado corretamente.

In [ ]:
# Visualize a 3x5 grid of random training images
helper_utils.visualise_images(train_dataset, grid=(3, 5))

* Crie uma nova instância do seu modelo `SimpleCNN`, desta vez configurada para todas as **15 classes**.

In [ ]:
# Get the number of classes
num_classes = len(train_dataset.classes)

# Instantiate the full model
model = SimpleCNN(num_classes)

# Print the model's architecture (notice, it now has 15 output classes)
print(model)

* Crie um novo otimizador `Adam` para o seu modelo completo de 15 classes.

In [ ]:
# Optimizer for the full model
optimizer = optim.Adam(model.parameters(), lr=0.001)

* Chame o `training_loop` para treinar o seu modelo de 15 classes por `25 epochs`. Em seguida, a função `plot_training_metrics` visualizará imediatamente as curvas de perda e acurácia desta rodada final de treinamento.

In [ ]:
# Start the training process for the full model on all 15 classes
trained_model, training_metrics = training_loop(
    model=model, 
    train_loader=train_loader, 
    val_loader=val_loader, 
    loss_function=loss_function, 
    optimizer=optimizer, 
    num_epochs=25, 
    device=device
)

# Visualize the training metrics for the full model
print("\n--- Training Plots ---\n")
helper_utils.plot_training_metrics(training_metrics)

Após treinar o modelo completo, você pode analisar os resultados. Mas espere, algo não está certo aqui. O seu modelo protótipo treinou com sucesso, mostrando uma melhoria constante. No entanto, o desempenho no conjunto de dados completo de 15 classes parece ter atingido um limite. O que aconteceu?

Um olhar atento aos gráficos revela o problema. Enquanto a **Perda de Treinamento** (*Training Loss*) diminui de forma consistente, a **Perda de Validação** (*Validation Loss*) cai por um tempo e, em seguida, começa a subir e a flutuar. Ao mesmo tempo, a **Acurácia de Validação** (*Validation Accuracy*) trava, atingindo um platô sem fazer progresso significativo adicional. Este é um caso clássico de **overfitting** (sobreajuste).

O *overfitting* ocorre quando um modelo aprende os dados de treinamento *bem até demais*, incluindo seus ruídos e peculiaridades específicas, em vez de aprender os padrões gerais subjacentes que o ajudariam a ter um bom desempenho em dados novos e não vistos. A lacuna cada vez maior entre a sua perda de treinamento e a de validação é um sinal claro de que o seu modelo está memorizando o conjunto de treinamento em vez de aprender a **generalizar**.

Você pode se perguntar por que isso aconteceu agora e não com o protótipo de 9 classes. O motivo é o aumento significativo na **complexidade da tarefa**. Distinguir entre 15 classes é muito mais difícil do que 9, exigindo que o modelo aprenda características mais sutis. Diante desse desafio mais difícil, o seu poderoso modelo CNN encontrou um caminho mais fácil para diminuir a perda de treinamento: ele começou a memorizar os dados de treinamento em vez de aprender a generalizar.

Este problema de *overfitting* apresenta um desafio realista, semelhante ao que você encontraria em um projeto do mundo real. Na tarefa avaliada deste módulo, você enfrentará esse problema fazendo várias atualizações em todo o seu pipeline para ver se consegue melhorar a capacidade de generalização do modelo.

In [ ]:
# ### Optional: Uncomment and run this cell to see the predictions made by the full model

# helper_utils.visualise_predictions(
#     model=trained_model, 
#     data_loader=val_loader, 
#     device=device, 
#     grid=(3, 5)
# )

## Conclusão

Parabéns por concluir o laboratório! Você navegou com sucesso por todo o pipeline de aprendizado de máquina, desde a preparação de dados até a construção, treinamento e análise da sua própria Rede Neural Convolucional.

Você colocou a teoria em prática construindo uma arquitetura CNN capaz de aprender padrões visuais complexos. Mais importante ainda, você vivenciou um fluxo de trabalho de desenvolvimento realista e iterativo, criando primeiro um protótipo bem-sucedido e, em seguida, aumentando a escala para um modelo mais complexo. Esse processo o levou a encontrar e diagnosticar o *overfitting* (sobreajuste), um desafio fundamental que todo profissional de aprendizado de máquina deve aprender a resolver.

As habilidades que você desenvolveu aqui o prepararam para o próximo passo. Você identificou o problema e, na tarefa avaliada, terá a chance de resolvê-lo. Bom trabalho!